# 🗃️ 벡터 데이터베이스: 교육용 RAG의 핵심

이 노트북에서는 벡터 데이터베이스를 사용하는 방법을 보여 줍니다.

Milvus는 이미 배포했으니, 이제 문서를 수집(ingest)하면서 어떻게 작동하는지 살펴보겠습니다! 🚀

## 📚 라이브러리 가져오기

벡터 데이터베이스 작업에 사용할 도구를 가져옵니다.

In [ ]:
# 벡터 데이터베이스 작업용 라이브러리 가져오기
from pymilvus import connections, utility, Collection, CollectionSchema, FieldSchema, DataType
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

## 🗃️ 벡터 데이터베이스 채우기

Milvus에 연결하고, 강의 콘텐츠를 검색 가능한 벡터로 저장할 컬렉션을 설정해 보겠습니다.

384차원 벡터를 생성하는 `all-MiniLM-L6-v2` 임베딩 모델을 사용합니다. Hugging Face에는 다양한 임베딩 모델이 있으며, **[Embedding LLM Leaderboard](https://huggingface.co/spaces/mteb/leaderboard)**에서 비교해 볼 수 있습니다. 단, 여러 모델의 임베딩을 섞어서 사용할 수는 없으니 주의하세요!

`all-MiniLM-L6-v2`가 최고 성능의 모델은 아니지만 같은 크기의 모델 중에서는 손꼽히는 모델이며, 이번 실습에서 빠르게 다운로드하고 실행할 수 있습니다.

## 🔗 Milvus 데이터베이스에 연결하기

‼️⚠️ 중요 ⚠️‼️

전달받은 사용자 이름과 클러스터 도메인을 입력하세요. 이 코드는 `{username}-canopy` 네임스페이스에 있는 Milvus 인스턴스에 연결합니다.

### 🖼️ Milvus Attu 웹 인터페이스 살펴보기

Milvus를 다루는 동안 Milvus Attu 웹 인터페이스도 함께 확인하세요. 혹시 창을 닫았다면 URL은 다음과 같습니다.

**Attu URL**: `https://milvus-attu-{username}-canopy.{cluster_domain}`

`{username}`과 `{cluster_domain}`을 본인의 값으로 바꾸세요. 처음에는 빈 데이터베이스가 보입니다. 시작 상태를 이해하기에 딱 좋습니다!

In [ ]:
# 중요! 여기에 사용자 이름과 클러스터 도메인을 입력하세요
username = "<USER_NAME>"
cluster_domain = "<CLUSTER_DOMAIN>"

# 교육 콘텐츠 벡터를 저장할 컬렉션 이름 정의
collection_name = "vectordb_collection"

connections.connect(
    uri=f"http://milvus.{username}-canopy.svc.cluster.local:19530",
    alias="default"
)

## 🧹 이전 실행 결과 정리하기

처음부터 시작할 수 있도록 기존 컬렉션을 삭제합니다.

In [ ]:
# 기존 컬렉션이 있으면 삭제
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

print(f"Collection list after cleanup: {utility.list_collections()}")

## 📋 데이터베이스 스키마 정의하기

벡터 데이터베이스에도 기존 데이터베이스처럼 스키마가 필요합니다. 우리의 스키마는 각 레코드에 고유 ID와 벡터 임베딩이 들어 있다고 정의합니다.

이 구조는 벡터 임베딩을 효율적으로 저장하고 관리하는 데 필수적입니다. Milvus는 어떤 필드가 들어오는지, 각 필드의 데이터 타입이 무엇인지 정확히 알아야 합니다.

In [ ]:
# 레코드를 고유하게 식별하기 위한 기본 키 필드 정의
id_field = FieldSchema(
    name="id",
    dtype=DataType.INT64,
    is_primary=True,
    auto_id=False
)

# 임베딩 모델과 출력 차원 지정
embedding_model = "all-MiniLM-L6-v2"  # Hugging Face 모델 이름
embedding_dim = 384  # 벡터 크기는 모델 출력과 일치해야 함

# 임베딩 값을 담을 벡터 필드 정의
embedding_field = FieldSchema(
    name="embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=embedding_dim
)

# ID 필드와 임베딩 필드를 묶어 컬렉션 스키마 구성
schema = CollectionSchema(
    fields=[id_field, embedding_field],
    description="Educational content vectors",
    enable_dynamic_field=False  # 엄격한 스키마 적용
)

## 🏗️ 컬렉션 만들기

Milvus의 컬렉션은 기존 데이터베이스의 테이블과 비슷합니다. 임베딩 벡터가 저장되고, 인덱싱되고, 조회되는 곳입니다.

교육 콘텐츠를 검색할 때 항상 최신 데이터를 얻을 수 있도록 strong consistency(강한 일관성)로 설정합니다.

In [ ]:
# 스키마와 설정으로 Milvus 컬렉션 생성
collection = Collection(
    name=collection_name, 
    schema=schema, 
    using='default',  # 기본 연결 사용
    shards_num=2,  # 데이터 분산을 위한 샤드 수
    consistency_level="Strong"  # 항상 최신 데이터를 반환하도록 보장
)

print(f"Collection: {collection.schema}\n")

print(f"Collection list: {utility.list_collections()}")

## 💾 벡터 데이터베이스에 콘텐츠 저장하기

이제 강의 콘텐츠 벡터를 Milvus에 저장해 검색할 수 있게 만들 차례입니다!

In [ ]:
# Hugging Face에서 임베딩 모델 불러오기
model = SentenceTransformer(embedding_model)

sentences = ["Introduction to Machine Learning covers supervised learning algorithms.",
             "Machine Learning fundamentals include supervised algorithm techniques.",
             "Computer Science department offers advanced database systems courses.",
             "Students can access research databases through the library portal."]

embeddings = model.encode(sentences)

In [ ]:
# 데이터베이스에 넣을 교육 콘텐츠 벡터 준비
data = [
    {"id": i, "embedding": vec.tolist()}  # numpy 배열을 리스트로 변환
    for i, vec in enumerate(embeddings)
]

# Milvus 컬렉션에 벡터 삽입
collection.insert(data=data)

# 빠른 유사도 검색을 위한 인덱스 생성
# COSINE 지표는 의미적 유사도에 매우 적합함
collection.create_index(
    field_name="embedding",
    index_params={
        "metric_type": "COSINE",  # 의미 검색을 위한 코사인 유사도
        "index_type": "IVF_FLAT",  # 역파일(Inverted file) 인덱스
        "params": {"nlist": 128}  # 인덱싱에 사용할 클러스터 수
    },
    index_name="idx"
)

# 변경 사항을 커밋하고 검색을 위해 컬렉션을 메모리에 로드
collection.flush()  # 데이터가 디스크에 기록되도록 보장
collection.load()   # 빠른 조회를 위해 컬렉션을 메모리에 로드

### 📊 삽입 후 Attu 확인하기

이제 Attu 웹 인터페이스에 다시 접속해 데이터 시각화를 확인해 보세요.

**Attu URL**: `https://milvus-attu-<USER_NAME>-canopy.<CLUSTER_DOMAIN>`

왼쪽의 Database 아이콘을 클릭하면 교육 콘텐츠 벡터 4개가 들어 있는 컬렉션이 보입니다.

![milvus-attu-database.png](milvus-attu-database.png)

## 🔎 벡터 데이터베이스 검색하기

이제 가장 흥미로운 부분입니다! 의미적 유사도를 이용해 교육 콘텐츠를 검색해 보겠습니다. 강의 관련 문장 4개를 저장했는데, 2개는 Machine Learning에 관한 문장이고 2개는 다른 주제입니다.

"What AI ethics topics are covered in the curriculum?"으로 검색하여 Machine Learning 콘텐츠가 관련 없는 학술 콘텐츠보다 더 높은 점수를 받는지 확인해 봅니다. 학생이 질문할 때 Canopy가 관련 강의 자료를 찾는 방식이 바로 이것입니다!

In [ ]:
# 교육 관련 질의로 의미 검색 시연
print("\n🔁 벡터 데이터베이스 검색 데모")
query = "I want to learn more about machine learning. Which classes should I take?"  # <- 머신러닝에 대해 더 배우고 싶어요. 어떤 수업을 들어야 할까요?
query_vector = model.encode([query])  # 질의를 벡터로 변환

# 벡터 데이터베이스에서 의미적으로 비슷한 콘텐츠 검색
results = collection.search(
    data=query_vector,
    anns_field="embedding",  # 벡터가 들어 있는 필드
    param={"metric_type": "COSINE"},  # 코사인 유사도 사용
    limit=3,  # 상위 3개 결과 반환
    output_fields=["embedding"]  # 결과에 원본 벡터 포함
)

# 결과 ID를 원본 텍스트에 매핑
id_to_text = {i: sentence for i, sentence in enumerate(sentences)}

# 검색 결과를 유사도 점수와 함께 표시
print(f"\n📌 Query: '{query}'\n")
print("📥 가장 유사한 검색 결과:\n")
for match in results[0]:
    match_id = match.id
    score = match.score  # 코사인 유사도 점수(높을수록 더 비슷함)
    matched_text = id_to_text.get(match_id, "[Unknown]")

    print(f"🆔 ID: {match_id}")
    print(f"🧠 텍스트: {matched_text}")
    print(f"📏 유사도 점수: {score:.4f}\n")

print("✅ 머신러닝 관련 콘텐츠가 더 높은 유사도 점수를 받는 것을 확인하세요!")
print("🎓 이것이 바로 Canopy가 관련 강의 자료를 찾아내는 방식입니다!")

In [ ]:
# 정리 - 연결을 닫고 컬렉션 삭제(선택 사항)
collection.release()
#utility.drop_collection(collection_name) # 이 줄의 주석을 해제해 실행하면 Attu에서 컬렉션이 보이지 않습니다

## 🎉 벡터 데이터베이스 설정을 완료했습니다!

**지금까지 한 일:**
- 배포된 Milvus 인스턴스에 연결했습니다
- 임베딩 모델이 텍스트를 검색 가능한 벡터로 변환하는 방법을 배웠습니다
- 교육 콘텐츠용 데이터베이스 스키마와 컬렉션을 만들었습니다  
- 강의 자료로 의미적 유사도를 테스트했습니다

실습 안내 문서로 돌아가 문서를 더 잘 처리하는 방법을 알아보세요.